This notebook serves as a rough proof of concept where each step is executed manually, in order to test the technology before comitting it to a proper Python project structure.

Import the packages

In [ ]:
import time #Added for timing the imports
print("Starting imports...")
start_time = time.time()
import cv2
print(f"Imported OpenCV: {cv2.__version__} (Time taken: {time.time() - start_time:.2f} seconds)")
import numpy as np
print(f"Imported NumPy: {np.__version__} (Time taken: {time.time() - start_time:.2f} seconds)")
from IPython.display import display, clear_output
import IPython.display as ipd
print(f"Imported IPython.display (Time taken: {time.time() - start_time:.2f} seconds)")
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.vision import drawing_utils as mp_drawing
import mediapipe as mp
print(f"Imported MediaPipe: {mp.__version__} (Time taken: {time.time() - start_time:.2f} seconds)")
from turtle import Screen, Turtle
print(f"Imported turtle (Time taken: {time.time() - start_time:.2f} seconds)")
import cv2 as cv
print(f"Imported cv2 as cv (Time taken: {time.time() - start_time:.2f} seconds)")


total_time = time.time() - start_time
print(f"Total time taken for imports: {total_time:.2f} seconds")

Starting imports...
Imported OpenCV: 4.13.0 (Time taken: 0.08 seconds)
Imported NumPy: 2.5.0 (Time taken: 0.08 seconds)
Imported IPython.display (Time taken: 0.08 seconds)
Imported MediaPipe: 0.10.35 (Time taken: 0.69 seconds)
Total time taken for imports: 0.69 seconds


Some sanity checks to ensure previous erroneous runs get cleaned up properly.

In [2]:
try:
    
    camera.release()
    print("Cleaned up camera")
except NameError:
    print("No camera to clean up")
    pass

No camera to clean up


Open the camera

In [3]:
camera = cv2.VideoCapture(0)

Configure stream

In [4]:
frame_width = int(camera.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(camera.get(cv2.CAP_PROP_FRAME_HEIGHT))

Display stream in notebook

In [5]:
def stream_camera():
    if not camera.isOpened():
        print("Error: Could not open camera.")
    try:
        while True:
            ret, frame = camera.read()
            if not ret:
                print("Error: Could not read frame.")
                break
            frame = cv2.resize(frame, (frame_width, frame_height))
            _, buffer = cv2.imencode('.jpg', frame)
            img_data = buffer.tobytes()
            clear_output(wait=True)
            display(ipd.Image(data=img_data))
    except KeyboardInterrupt:
        pass
    except Exception as e:
        print(f"An error occurred: {e}")    

Import hand classifier

In [6]:
baseOptions = python.BaseOptions(model_asset_path="hand_landmarker.task")
options = vision.HandLandmarkerOptions(base_options=baseOptions, num_hands=2)
detector = vision.HandLandmarker.create_from_options(options)

Display stream in notebook, this time with bounding boxes.


In [ ]:
def stream_camera_with_detection():
    if not camera.isOpened():
        print("Error: Could not open camera.")
    try:
        while True:
            ret, frame = camera.read()
            if not ret:
                print("Error: Could not read frame.")
                break
            frame = cv2.resize(frame, (frame_width, frame_height))
            rgb_frame = cv.cvtColor(frame, cv.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
            result = detector.detect(mp_image)
            if result.hand_landmarks:
                for hand_landmarks in result.hand_landmarks:

                    mp_drawing.draw_landmarks(frame, hand_landmarks, vision.HandLandmarksConnections.HAND_CONNECTIONS)
            cv2.imshow('Hand Detection', frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
    except KeyboardInterrupt:
        pass


# stream_camera_with_detection()

Calculate a "center of mass" by grabbing the average position of the hands

In [ ]:
def stream_camera_with_detection_center_of_mass():
    if not camera.isOpened():
        print("Error: Could not open camera.")
    try:
        while True:
            ret, frame = camera.read()
            if not ret:
                print("Error: Could not read frame.")
                break
            frame = cv2.resize(frame, (frame_width, frame_height))
            rgb_frame = cv.cvtColor(frame, cv.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
            result = detector.detect(mp_image)
            x_list = []
            y_list = []
            if result.hand_landmarks:
                for hand_landmarks in result.hand_landmarks:
                    x_list.extend([lm.x for lm in hand_landmarks])
                    y_list.extend([lm.y for lm in hand_landmarks])
                    mp_drawing.draw_landmarks(frame, hand_landmarks, vision.HandLandmarksConnections.HAND_CONNECTIONS)
            average_x = int(np.mean(x_list) * frame_width) if x_list else 0
            average_y = int(np.mean(y_list) * frame_height) if y_list else 0
            cv2.circle(frame, (average_x, average_y), 10, (255,0,255), -1)
            # frame = cv2.flip(frame, 1)  # Flip the frame horizontally for a mirror effect
            cv2.rectangle(frame, (average_x + 15, average_y+5), (average_x + 150, average_y - 20), (255,255,255), cv2.FILLED)
            cv2.putText(frame, f"Avg: ({average_x}, {average_y})", (average_x + 15, average_y), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,0,255), 1)
            cv2.putText(frame, f"FPS: {1/(time.time() - start_time):.2f}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,0), 2)
            cv2.imshow('Hand Detection', frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
    except KeyboardInterrupt:
        pass


# stream_camera_with_detection_center_of_mass()

Translate center of mass to position

In [ ]:
def stream_camera_with_detection_center_of_mass():
    if not camera.isOpened():
        print("Error: Could not open camera.")
    try:
        max_x = frame_width
        max_y = frame_height
        while True:
            average_x = max_x // 2
            average_y = max_y // 2
            ret, frame = camera.read()
            if not ret:
                print("Error: Could not read frame.")
                break
            frame = cv2.resize(frame, (frame_width, frame_height))
            rgb_frame = cv.cvtColor(frame, cv.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
            result = detector.detect(mp_image)
            x_list = []
            y_list = []
            if result.hand_landmarks:
                for hand_landmarks in result.hand_landmarks:
                    x_list.extend([lm.x for lm in hand_landmarks])
                    y_list.extend([lm.y for lm in hand_landmarks])
                    mp_drawing.draw_landmarks(frame, hand_landmarks, vision.HandLandmarksConnections.HAND_CONNECTIONS)
                average_x = int(np.mean(x_list) * frame_width)  
                average_y = int(np.mean(y_list) * frame_height) 
                cv2.circle(frame, (average_x, average_y), 10, (255,0,255), -1)
                # frame = cv2.flip(frame, 1)  # Flip the frame horizontally for a mirror effect
                cv2.rectangle(frame, (average_x + 15, average_y+5), (average_x + 150, average_y - 20), (255,255,255), cv2.FILLED)
                cv2.putText(frame, f"Avg: ({average_x}, {average_y})", (average_x + 15, average_y), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,0,255), 1)
            x_pos = "center"
            y_pos = "center"
            if average_x > 0.6*max_x:
                x_pos = "right"
            if average_x < 0.4*max_x:
                x_pos = "left"
            if average_y > 0.6*max_y:
                y_pos = "bottom"
            if average_y < 0.4*max_y:
                y_pos = "top"
            cv2.putText(frame, f"Position: {x_pos}, {y_pos}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,0), 2)
            cv2.imshow('Hand Detection', frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
    except KeyboardInterrupt:
        pass


# stream_camera_with_detection_center_of_mass()

Let's put this on a simple turtle

In [33]:
from turtle import Screen, Turtle

def stream_camera_with_detection_center_of_mass():
    screen = Screen()
    screen.title("Tutle")
    screen.setup(width=frame_width, height=frame_height)
    turtle = Turtle()
    if not camera.isOpened():
        print("Error: Could not open camera.")
    try:
        max_x = frame_width
        max_y = frame_height
        while True:
            average_x = max_x // 2
            average_y = max_y // 2
            ret, frame = camera.read()
            if not ret:
                print("Error: Could not read frame.")
                break
            frame = cv2.resize(frame, (frame_width, frame_height))
            rgb_frame = cv.cvtColor(frame, cv.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
            result = detector.detect(mp_image)
            x_list = []
            y_list = []
            if result.hand_landmarks:
                for hand_landmarks in result.hand_landmarks:
                    x_list.extend([lm.x for lm in hand_landmarks])
                    y_list.extend([lm.y for lm in hand_landmarks])
                    mp_drawing.draw_landmarks(frame, hand_landmarks, vision.HandLandmarksConnections.HAND_CONNECTIONS)
                average_x = int(np.mean(x_list) * frame_width)  
                average_y = int(np.mean(y_list) * frame_height) 
                cv2.circle(frame, (average_x, average_y), 10, (255,0,255), -1)
                # frame = cv2.flip(frame, 1)  # Flip the frame horizontally for a mirror effect
                cv2.rectangle(frame, (average_x + 15, average_y+5), (average_x + 150, average_y - 20), (255,255,255), cv2.FILLED)
                cv2.putText(frame, f"Avg: ({average_x}, {average_y})", (average_x + 15, average_y), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,0,255), 1)
            x_pos = "center"
            y_pos = "center"
            if average_x > 0.6*max_x:
                turtle.right(10)
                x_pos = "right"
            if average_x < 0.4*max_x:
                turtle.left(10)
                x_pos = "left"
            if average_y > 0.6*max_y:
                turtle.backward(10)
                y_pos = "bottom"
            if average_y < 0.4*max_y:
                turtle.forward(10)
                y_pos = "top"
            cv2.putText(frame, f"Position: {x_pos}, {y_pos}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,0), 2)
            cv2.imshow('Hand Detection', frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
    except KeyboardInterrupt:
        pass


stream_camera_with_detection_center_of_mass()

Let's make the movement controls less "all or nothing", and have the steering / accelleration depend on distance from the center

In [38]:
def stream_camera_with_detection_center_of_mass():
    screen = Screen()
    screen.title("Tutle")
    screen.setup(width=frame_width, height=frame_height)
    turtle = Turtle()
    if not camera.isOpened():
        print("Error: Could not open camera.")
    try:
        max_x = frame_width
        max_y = frame_height
        while True:
            average_x = max_x // 2
            average_y = max_y // 2
            ret, frame = camera.read()
            if not ret:
                print("Error: Could not read frame.")
                break
            frame = cv2.resize(frame, (frame_width, frame_height))
            rgb_frame = cv.cvtColor(frame, cv.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
            result = detector.detect(mp_image)
            x_list = []
            y_list = []
            if result.hand_landmarks:
                for hand_landmarks in result.hand_landmarks:
                    x_list.extend([lm.x for lm in hand_landmarks])
                    y_list.extend([lm.y for lm in hand_landmarks])
                    mp_drawing.draw_landmarks(frame, hand_landmarks, vision.HandLandmarksConnections.HAND_CONNECTIONS)
                average_x = int(np.mean(x_list) * frame_width)  
                average_y = int(np.mean(y_list) * frame_height) 
                cv2.circle(frame, (average_x, average_y), 10, (255,0,255), -1)
                # frame = cv2.flip(frame, 1)  # Flip the frame horizontally for a mirror effect
                cv2.rectangle(frame, (average_x + 15, average_y+5), (average_x + 150, average_y - 20), (255,255,255), cv2.FILLED)
                cv2.putText(frame, f"Avg: ({average_x}, {average_y})", (average_x + 15, average_y), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,0,255), 1)
            x_pos = "center"
            y_pos = "center"
            if average_x > 0.6*max_x:
                distance = average_x - 0.6*max_x
                turtle.right(10 * (distance / (0.4*max_x)))
                x_pos = "right"
            if average_x < 0.4*max_x:
                distance = 0.4*max_x - average_x
                turtle.left(10 * (distance / (0.4*max_x)))
                x_pos = "left"
            if average_y > 0.6*max_y:
                distance = average_y - 0.6*max_y

                turtle.backward(10 * (distance / (0.4*max_y)))
                y_pos = "bottom"
            if average_y < 0.4*max_y:
                distance = 0.4*max_y - average_y
                turtle.forward(10 * (distance / (0.4*max_y)))
                y_pos = "top"
            cv2.putText(frame, f"Position: {x_pos}, {y_pos}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,0), 2)
            cv2.imshow('Hand Detection', frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
    except KeyboardInterrupt:
        pass


stream_camera_with_detection_center_of_mass()